# Phase 4: Counterfactual Search Optimizer
In this notebook, we utilize the Cox Proportional Hazards model and the Latent Manifold Autoencoder to find biologically plausible counterfactual paths for high-risk patients. We use a PyTorch-native gradient descent optimizer operating in the latent space.


In [ ]:
import torch
import pandas as pd
import numpy as np
import joblib
import os
import sys

sys.path.append('../src')
from optimizer import CounterfactualOptimizer
from data_pipeline import load_and_preprocess_data

# Load Data
print("Loading data...")
train_df, test_df, imm, dir_f, act = load_and_preprocess_data(filepath='../data/frmgham2.csv', random_seed=42)

# Load configuration
ae_config = joblib.load('../models/ae_config.joblib')
feature_cols = ae_config['feature_cols']

# Initialize Optimizer
opt = CounterfactualOptimizer(models_dir='../models')
scaler = opt.scaler
cont_cols = opt.cont_cols
cont_idx = opt.cont_idx
bin_cols = opt.bin_cols
bin_idx = opt.bin_idx


## 1. Selecting High-Risk Patients
We identify a subset of high-risk patients from the test cohort (e.g., those who experienced a cardiac event) to demonstrate the counterfactual search.


In [ ]:
# Filter to those who had an event in the test set
test_events = test_df[test_df['ANYCHD'] == 1].copy()

# Pick 3 original patients as case studies
sample_patients = test_events.head(3)

# Find a patient with very low SYSBP to test the hard bound
sysbp_scale_idx = cont_cols.index('SYSBP')
sysbp_thresh = (100 - scaler.mean_[sysbp_scale_idx]) / scaler.scale_[sysbp_scale_idx]
low_sysbp_patient = test_df[test_df['SYSBP'] < sysbp_thresh].head(1)

# Find a patient with low TOTCHOL to test the new monotonicity rule
totchol_scale_idx = cont_cols.index('TOTCHOL')
totchol_thresh = (140 - scaler.mean_[totchol_scale_idx]) / scaler.scale_[totchol_scale_idx]
low_totchol_patient = test_df[test_df['TOTCHOL'] < totchol_thresh].head(1)

# Combine them all
sample_patients = pd.concat([sample_patients, low_sysbp_patient, low_totchol_patient])
patient_vectors = sample_patients[feature_cols].values
print(f"Selected {len(patient_vectors)} high-risk patients for demonstration.")


## 2. Running the Counterfactual Search
For each patient, the optimizer minimizes the Cox log partial hazard while enforcing proximity, immutability, and hard clinical bounds.


In [ ]:
results = []

for i, p_vec in enumerate(patient_vectors):
    res = opt.optimize(p_vec)
    
    # We want to format the output nicely. Let's unscale the continuous features.
    x_orig_cont = res['x_orig'][cont_idx]
    x_cf_cont = res['x_cf'][cont_idx]
    
    # Inverse transform requires 2D array
    x_orig_native = scaler.inverse_transform(x_orig_cont.reshape(1, -1))[0]
    x_cf_native = scaler.inverse_transform(x_cf_cont.reshape(1, -1))[0]
    
    # Format changes
    changes = []
    
    # Continuous changes
    for feat, orig_val, cf_val in zip(cont_cols, x_orig_native, x_cf_native):
        diff = cf_val - orig_val
        if abs(diff) > 0.5: # Only show meaningful changes
            changes.append(f"{feat}: {orig_val:.1f} -> {cf_val:.1f} ({diff:+.1f})")
            
    # Binary changes
    x_orig_bin = res['x_orig'][bin_idx]
    x_cf_bin = np.round(res['x_cf'][bin_idx])
    for feat, orig_val, cf_val in zip(bin_cols, x_orig_bin, x_cf_bin):
        diff = cf_val - orig_val
        if diff != 0:
            changes.append(f"{feat}: {int(orig_val)} -> {int(cf_val)}")
            
    results.append({
        'Patient ID': f"Patient {i+1}",
        'Baseline Log Hazard': f"{res['baseline_hazard']:.3f}",
        'Post-Recourse Log Hazard': f"{res['final_hazard']:.3f}",
        'Hazard Drop': f"{res['hazard_drop']:.3f}",
        'Success': res['success'],
        'Suggested Interventions': " | ".join(changes) if (changes and res['success']) else "No feasible recourse found."
    })

pd.set_option('display.max_colwidth', None)
res_df = pd.DataFrame(results)
display(res_df)
